# Phase 6 — Leakage-Safe Walk-Forward Validation and Final Holdout

**What:** create expanding-window validation folds plus one untouched final test period.

**Why:** the project is future forecasting. We need to evaluate on later periods without shuffling future observations into training.

**Decision:** use target-time boundaries for every fold; use 2014, 2015, and 2016 as successive validation periods; keep January–February 2017 completely untouched for the final evaluation.

**Key leakage controls:**
- No random shuffle.
- A row belongs to a split according to its `target_datetime`.
- Training targets never belong to a later validation/test period.
- Lag/rolling features are created causally within station/time segments.
- Model preprocessing is fitted inside each fold using training rows only.
- Final test data is not used for model, feature, or hyperparameter decisions.

## 1. Load engineered features and define the supervised rows

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src" / "project_config.py").is_file():
        PROJECT_ROOT = candidate
        break
sys.path.insert(0, str(PROJECT_ROOT))
from src.project_config import *

import pandas as pd
import numpy as np

if not FEATURE_DATA.exists():
    raise FileNotFoundError("Run 05_feature_engineering.ipynb first.")

df = pd.read_csv(FEATURE_DATA, parse_dates=["datetime", "target_datetime"])
df = df.sort_values(["station", "datetime"]).reset_index(drop=True)

HISTORY_FEATURES = (
    [f"PM2.5_lag_{x}h" for x in LAGS]
    + [f"PM2.5_roll_mean_{x}h" for x in ROLL_MEANS]
    + [f"PM2.5_roll_std_{x}h" for x in ROLL_STDS]
)
REQUIRED_HISTORY = ["PM2.5"] + HISTORY_FEATURES

# Only keep rows with an observed next-hour target and complete temporal history.
model_ready = df[TARGET].notna() & df[TARGET_TIME].notna() & df[REQUIRED_HISTORY].notna().all(axis=1)
model_df = df.loc[model_ready].copy()

print("Raw engineered rows:", f"{len(df):,}")
print("Model-ready supervised rows:", f"{len(model_df):,}")
print("Rows removed for missing target/history:", f"{len(df)-len(model_df):,}")

Raw engineered rows: 420,768
Model-ready supervised rows: 359,177
Rows removed for missing target/history: 61,591


## 2. Build expanding-window folds and the final holdout

In [2]:
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

def target_mask(frame, start=None, end=None):
    mask = frame[TARGET_TIME].notna()
    if start is not None:
        mask &= frame[TARGET_TIME] >= pd.Timestamp(start)
    if end is not None:
        mask &= frame[TARGET_TIME] <= pd.Timestamp(end)
    return mask

fold_records = []

for fold in WALK_FORWARD_FOLDS:
    train_mask = target_mask(model_df, fold["train_start"], fold["train_end"])
    valid_mask = target_mask(model_df, fold["validation_start"], fold["validation_end"])

    train_part = model_df.loc[train_mask].copy()
    valid_part = model_df.loc[valid_mask].copy()

    assert len(train_part) > 0 and len(valid_part) > 0
    assert train_part[TARGET_TIME].max() < valid_part[TARGET_TIME].min()
    assert not set(train_part.index).intersection(valid_part.index)

    train_path = SPLIT_DIR / f"{fold['name']}_train_{fold['train_start'][:4]}_to_{fold['train_end'][:4]}.csv"
    valid_path = SPLIT_DIR / f"{fold['name']}_validation_{fold['validation_start'][:4]}.csv"
    train_part.to_csv(train_path, index=False)
    valid_part.to_csv(valid_path, index=False)

    fold_records.append([
        fold["name"], "walk_forward", len(train_part), len(valid_part),
        train_part[TARGET_TIME].min(), train_part[TARGET_TIME].max(),
        valid_part[TARGET_TIME].min(), valid_part[TARGET_TIME].max(),
        train_path.name, valid_path.name
    ])

# Final untouched 2017 holdout. Nothing below uses this file for tuning.
test_mask = target_mask(model_df, FINAL_TEST_TARGET_START, FINAL_TEST_TARGET_END)
final_test = model_df.loc[test_mask].copy()
assert len(final_test) > 0
assert final_test[TARGET_TIME].min() >= pd.Timestamp(FINAL_TEST_TARGET_START)

test_path = SPLIT_DIR / "final_test_2017_01_to_02.csv"
final_test.to_csv(test_path, index=False)

manifest = pd.DataFrame(
    fold_records,
    columns=[
        "fold", "type", "train_rows", "validation_rows",
        "train_target_start", "train_target_end",
        "validation_target_start", "validation_target_end",
        "train_file", "validation_file"
    ]
)

display(manifest)
manifest.to_csv(SPLIT_DIR / "walk_forward_manifest.csv", index=False)

test_manifest = pd.DataFrame([{
    "split": "final_test",
    "rows": len(final_test),
    "target_start": final_test[TARGET_TIME].min(),
    "target_end": final_test[TARGET_TIME].max(),
    "file": test_path.name,
    "protected": True
}])
display(test_manifest)
test_manifest.to_csv(SPLIT_DIR / "final_test_manifest.csv", index=False)

,fold,type,train_rows,validation_rows,train_target_start,train_target_end,validation_target_start,validation_target_end,train_file,validation_file
0,fold_1,walk_forward,82553,87927,2013-03-02 01:00:00,2013-12-31 23:00:00,2014-01-01,2014-12-31 23:00:00,fold_1_train_2013_to_2013.csv,fold_1_validation_2014.csv
1,fold_2,walk_forward,170480,87088,2013-03-02 01:00:00,2014-12-31 23:00:00,2015-01-01,2015-12-31 23:00:00,fold_2_train_2013_to_2014.csv,fold_2_validation_2015.csv
2,fold_3,walk_forward,257568,86826,2013-03-02 01:00:00,2015-12-31 23:00:00,2016-01-01,2016-12-31 23:00:00,fold_3_train_2013_to_2015.csv,fold_3_validation_2016.csv


,split,rows,target_start,target_end,file,protected
0,final_test,14783,2017-01-01,2017-02-28 23:00:00,final_test_2017_01_to_02.csv,True


## 3. Verify leakage boundaries

In [3]:
# Validate that each training period ends before its validation period.
for fold in WALK_FORWARD_FOLDS:
    assert pd.Timestamp(fold["train_end"]) < pd.Timestamp(fold["validation_start"])
    assert pd.Timestamp(fold["validation_end"]) < pd.Timestamp(FINAL_TEST_TARGET_START)

# Validate that the final test is disjoint from every development period.
for fold in WALK_FORWARD_FOLDS:
    train_ids = set(model_df.index[target_mask(model_df, fold["train_start"], fold["train_end"])])
    valid_ids = set(model_df.index[target_mask(model_df, fold["validation_start"], fold["validation_end"])])
    test_ids = set(final_test.index)
    assert not train_ids.intersection(test_ids)
    assert not valid_ids.intersection(test_ids)

# A target at t+1 cannot be used in an earlier training period.
assert all(
    pd.Timestamp(fold["train_end"]) < pd.Timestamp(fold["validation_start"])
    for fold in WALK_FORWARD_FOLDS
)

print("LEAKAGE CHECK: PASS")
print("Development validation periods:", [f["validation_start"][:4] for f in WALK_FORWARD_FOLDS])
print("Final holdout: January-February 2017")

LEAKAGE CHECK: PASS
Development validation periods: ['2014', '2015', '2016']
Final holdout: January-February 2017


## Split decision

The final test period is intentionally short because the source dataset ends on 28 February 2017. Rather than replacing the time-based strategy with random shuffling, the project uses **multiple expanding-window validation periods** to measure stability across several historical future periods.

The final 2017 holdout remains untouched until the project's final evaluation phase.